In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "BPQ_H.xpt"
csv_file = DATA_DIR / "BPQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 BPQ_H XPT -> CSV")
print("=" * 90)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BPQ020",
    "BPQ030",
    "BPD035",
    "BPQ040A",
    "BPQ050A",
    "BPQ056",
    "BPD058",
    "BPQ059",
    "BPQ080",
    "BPQ060",
    "BPQ070",
    "BPQ090D",
    "BPQ100D"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BPQ_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 6464:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 14:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,464 rows x 14 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Attached BPQ_H contains NO genuine decimal variables.
#
# All fields are:
#
#     SEQN
#     questionnaire response codes
#     whole-number ages
#     whole-number frequency/count fields
#
# Therefore all 14 variables can safely become Int64.
#
# IMPORTANT:
#
# This applies only to BPQ_H.
#
# Genuine Dietary/Lab/Examination decimals in other
# files should continue to remain exact floats.
#

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 14:

    raise ValueError(
        "Expected exactly 14 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached BPQ_H contains NONE.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 14 BPQ_H fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Tiny XPORT artifacts:",
    tiny_total
)

print(
    "Ordinary numeric zeros:",
    true_zero_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in BPQ_H."
    )


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary zeros "
        "found in BPQ_H."
    )


print(
    "PASS: BPQ_H requires no zero restoration."
)


# ============================================================
# 6. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run this BEFORE converting to Int64.
#
# np.round() is used ONLY to test whether values
# are whole-number.
#
# It does not alter the data.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine fractional variables unexpectedly "
        "found in BPQ_H. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83729:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83729 validated."
)


# ============================================================
# 8. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# Every BPQ_H variable is proven whole-number.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 14 BPQ_H fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 9. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        actual_missing
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 10. CDC CHECK - BPQ020
# ============================================================

#
# Ever told you had high blood pressure?
#
# 1 Yes         2,174
# 2 No          4,285
# 9 Don't know      5
#

check_frequency(
    "BPQ020",
    {
        1: 2174,
        2: 4285,
        9: 5
    },
    0
)


# ============================================================
# 11. CDC CHECK - BPQ030
# ============================================================

#
# Told had high BP on 2 or more visits?
#

check_frequency(
    "BPQ030",
    {
        1: 1738,
        2: 428,
        9: 8
    },
    4290
)


# ============================================================
# 12. CDC CHECK - BPD035
# ============================================================

#
# Age first told had hypertension.
#
# CDC:
#
# 10-79 = 2,100
# 80    =    35
# 777   =     1
# 999   =    28
# Missing = 4,300
#

s = df["BPD035"]


age_10_79 = int(
    s.between(
        10,
        79
    )
    .sum()
)


age_80 = int(
    s.eq(80)
    .sum()
)


refused_777 = int(
    s.eq(777)
    .sum()
)


dontknow_999 = int(
    s.eq(999)
    .sum()
)


missing = int(
    s.isna()
    .sum()
)


print("\n--- BPD035 ---")

print(
    "Age 10-79:",
    age_10_79
)

print(
    "Age 80+:",
    age_80
)

print(
    "Refused:",
    refused_777
)

print(
    "Don't know:",
    dontknow_999
)

print(
    "Missing:",
    missing
)


if not (
    age_10_79 == 2100
    and
    age_80 == 35
    and
    refused_777 == 1
    and
    dontknow_999 == 28
    and
    missing == 4300
):

    raise ValueError(
        "BPD035 differs from CDC release."
    )


print(
    "PASS: BPD035 matches CDC."
)


# ============================================================
# 13. CDC CHECK - BPQ040A
# ============================================================

check_frequency(
    "BPQ040A",
    {
        1: 1815,
        2: 357,
        9: 2
    },
    4290
)


# ============================================================
# 14. CDC CHECK - BPQ050A
# ============================================================

check_frequency(
    "BPQ050A",
    {
        1: 1586,
        2: 228,
        9: 1
    },
    4649
)


# ============================================================
# 15. HYPERTENSION ROUTING CHECK
# ============================================================

#
# BPQ030 and BPQ040A are populated exactly
# for BPQ020 = 1.
#

bpq020_yes = (
    df["BPQ020"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq030_present = (
    df["BPQ030"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq040a_present = (
    df["BPQ040A"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        bpq020_yes
        !=
        bpq030_present
    )
) != 0:

    raise ValueError(
        "BPQ030 routing does not match BPQ020=1."
    )


if int(
    np.sum(
        bpq020_yes
        !=
        bpq040a_present
    )
) != 0:

    raise ValueError(
        "BPQ040A routing does not match BPQ020=1."
    )


print(
    "PASS: BPQ020 hypertension routing validated."
)


# ============================================================
# 16. BPQ050A ROUTING CHECK
# ============================================================

#
# BPQ050A should be populated exactly when:
#
#     BPQ040A = 1
#

bpq040a_yes = (
    df["BPQ040A"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq050a_present = (
    df["BPQ050A"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        bpq040a_yes
        !=
        bpq050a_present
    )
) != 0:

    raise ValueError(
        "BPQ050A routing does not match BPQ040A=1."
    )


print(
    "PASS: BPQ050A medication routing validated."
)


# ============================================================
# 17. CDC CHECK - BPQ056
# ============================================================

#
# Took BP at home during last 12 months?
#

check_frequency(
    "BPQ056",
    {
        1: 1579,
        2: 4883,
        9: 2
    },
    0
)


# ============================================================
# 18. CDC CHECK - BPD058
# ============================================================

#
# How often checked BP at home.
#
# CDC:
#
# 1-2704 = 1,574
# 9999   =     5
# Missing = 4,885
#

s = df["BPD058"]


range_count = int(
    s.between(
        1,
        2704
    )
    .sum()
)


dontknow_count = int(
    s.eq(9999)
    .sum()
)


missing_count = int(
    s.isna()
    .sum()
)


if not (
    range_count == 1574
    and
    dontknow_count == 5
    and
    missing_count == 4885
):

    raise ValueError(
        "BPD058 differs from CDC release."
    )


print(
    "PASS: BPD058 range/count validated."
)


# ============================================================
# 19. BPD058 ROUTING CHECK
# ============================================================

#
# BPD058 is populated exactly for participants
# who answered BPQ056 = 1.
#

bpq056_yes = (
    df["BPQ056"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpd058_present = (
    df["BPD058"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        bpq056_yes
        !=
        bpd058_present
    )
) != 0:

    raise ValueError(
        "BPD058 routing does not match BPQ056=1."
    )


print(
    "PASS: Home BP monitoring routing validated."
)


# ============================================================
# 20. CDC CHECK - BPQ059
# ============================================================

check_frequency(
    "BPQ059",
    {
        1: 862,
        2: 5601,
        7: 1
    },
    0
)


# ============================================================
# 21. CDC CHECK - BPQ080
# ============================================================

#
# Ever told blood cholesterol was high?
#

check_frequency(
    "BPQ080",
    {
        1: 2037,
        2: 4391,
        9: 36
    },
    0
)


# ============================================================
# 22. CDC CHECK - BPQ060
# ============================================================

#
# Ever had blood cholesterol checked?
#
# Participants with BPQ080=1 skip this question.
#

check_frequency(
    "BPQ060",
    {
        1: 2583,
        2: 1689,
        7: 3,
        9: 152
    },
    2037
)


# ============================================================
# 23. BPQ060 SKIP-PATTERN CHECK
# ============================================================

#
# CDC skip logic:
#
# BPQ080 = 1
#     skips BPQ060.
#
# Therefore missing BPQ060 should exactly match
# BPQ080 = 1.
#

bpq080_yes = (
    df["BPQ080"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq060_missing = (
    df["BPQ060"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        bpq080_yes
        !=
        bpq060_missing
    )
) != 0:

    raise ValueError(
        "BPQ060 missing pattern does not "
        "match BPQ080=1."
    )


print(
    "PASS: BPQ080/BPQ060 skip pattern validated."
)


# ============================================================
# 24. CDC CHECK - BPQ070
# ============================================================

check_frequency(
    "BPQ070",
    {
        1: 2998,
        2: 817,
        3: 535,
        4: 240,
        9: 30
    },
    1844
)


# ============================================================
# 25. CDC CHECK - BPQ090D
# ============================================================

check_frequency(
    "BPQ090D",
    {
        1: 1450,
        2: 3163,
        9: 7
    },
    1844
)


# ============================================================
# 26. CHOLESTEROL ROUTING CHECK
# ============================================================

#
# BPQ070 and BPQ090D are asked when participant:
#
#     BPQ080 = 1
#
# OR
#
#     BPQ060 = 1
#
# 2,037 + 2,583 = 4,620 records.
#

cholesterol_path = (
    (
        df["BPQ080"]
        .eq(1)
    )
    |
    (
        df["BPQ060"]
        .eq(1)
    )
)


cholesterol_path = (
    cholesterol_path
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq070_present = (
    df["BPQ070"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq090d_present = (
    df["BPQ090D"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        cholesterol_path
        !=
        bpq070_present
    )
) != 0:

    raise ValueError(
        "BPQ070 routing pattern differs."
    )


if int(
    np.sum(
        cholesterol_path
        !=
        bpq090d_present
    )
) != 0:

    raise ValueError(
        "BPQ090D routing pattern differs."
    )


print(
    "PASS: Cholesterol follow-up routing validated."
)


# ============================================================
# 27. CDC CHECK - BPQ100D
# ============================================================

check_frequency(
    "BPQ100D",
    {
        1: 1136,
        2: 310,
        9: 4
    },
    5014
)


# ============================================================
# 28. BPQ100D ROUTING CHECK
# ============================================================

#
# BPQ100D is populated exactly when:
#
#     BPQ090D = 1
#

bpq090d_yes = (
    df["BPQ090D"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq100d_present = (
    df["BPQ100D"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        bpq090d_yes
        !=
        bpq100d_present
    )
) != 0:

    raise ValueError(
        "BPQ100D routing does not match BPQ090D=1."
    )


print(
    "PASS: Cholesterol medication routing validated."
)


# ============================================================
# 29. CONFIRM NO ZERO VALUES WERE CREATED
# ============================================================

#
# Attached source contained:
#
# tiny artifacts = 0
# ordinary zeros = 0
#
# Therefore no zero should appear after cleaning.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO CHECK ---")

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values were created:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 30. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# BPQ_H requires NO numeric correction.
#
# Converting whole-number fields from:
#
#     1.0 -> 1
#     73557.0 -> 73557
#
# changes CSV formatting only.
#
# Numeric meaning remains identical.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 14 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 31. EXPORT CSV
# ============================================================

#
# Standard NHANES fidelity export:
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
# BPQ_H has no decimals, but maintaining this rule keeps
# your workflow safe for mixed datasets.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 32. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 33. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 14 CSV variables round-trip "
    "to exact source XPT values."
)


# ============================================================
# 34. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 35. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 36. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 37. RAW CSV .0 CHECK
# ============================================================

#
# BPQ_H has NO genuine decimal variables.
#
# Therefore it is safe to check all 14 fields.
#
# Examples:
#
#     SEQN
#     73557.0 -> 73557
#
#     BPQ020
#     1.0 -> 1
#
#     BPD035
#     45.0 -> 45
#
#     BPD058
#     365.0 -> 365
#
# Do NOT use a global .0 check in datasets that have
# genuine Dietary/Lab/Exam decimals.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any BPQ_H field."
)


# ============================================================
# 38. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO CHECK ---")

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found "
        "in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 39. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 40. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83729:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 41. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "BPQ_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - Blood Pressure & Cholesterol"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "BPQ020 high-BP yes:",
    int(
        df["BPQ020"]
        .eq(1)
        .sum()
    )
)


print(
    "BPQ050A currently taking HBP medicine:",
    int(
        df["BPQ050A"]
        .eq(1)
        .sum()
    )
)


print(
    "BPQ080 high-cholesterol yes:",
    int(
        df["BPQ080"]
        .eq(1)
        .sum()
    )
)


print(
    "BPQ100D currently taking cholesterol medicine:",
    int(
        df["BPQ100D"]
        .eq(1)
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 BPQ_H XPT -> CSV
Rows: 6,464
Columns: 14
PASS: Structure = 6,464 rows x 14 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
Whole-number/code fields: 14

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: BPQ_H requires no zero restoration.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
BPQ020: 0 fractional observations
BPQ030: 0 fractional observations
BPD035: 0 fractional observations
BPQ040A: 0 fractional observations
BPQ050A: 0 fractional observations
BPQ056: 0 fractional observations
BPD058: 0 fractional observations
BPQ059: 0 fractional observations
BPQ080: 0 fractional observations
BPQ060: 0 fractional observations
BPQ070: 0 fractional observations
BPQ090D: 0 fractional observations
BPQ100D: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 73557
Maximum: 83729
Missing: 0
Duplicates: 0
PASS: SEQN 73557-83729 validated.

PASS: All 14 BPQ_H fields converted to n